# hpc-as-api — Live Deployment Demo

End-to-end test notebook for the Gemma 4 31B deployment on Lakeshore (ga-002).  
Measures TTFT, throughput, and concurrent capacity.

**What you need:**
- `pip install openai httpx`
- Set `HPC_GATEWAY_URL` and `HPC_GATEWAY_KEY` as environment variables (or paste them in the config cell below)

**Models available:**
| Alias | Model | Node | Context |
|-------|-------|------|---------|
| `gemma4-31b` | google/gemma-4-31B-it | ga-002 (2× A100 80GB, TP2) | 128K |
| `qwen25-vl-72b` | Qwen/Qwen2.5-VL-72B-Instruct-AWQ | ghi2-002 (1× H100) | 64K |

In [ ]:
import os

# ── Configuration ─────────────────────────────────────────────────────────────
# Set HPC_GATEWAY_URL and HPC_GATEWAY_KEY as environment variables before
# running this notebook.  Never paste a real API key directly into this file.
BASE_URL = os.environ["HPC_GATEWAY_URL"]   # e.g. https://relay.stream.acer.uic.edu:8001/v1
API_KEY  = os.environ["HPC_GATEWAY_KEY"]   # e.g. sk-stream-...
MODEL    = os.getenv("HPC_GATEWAY_MODEL", "gemma4-31b")

print(f"Endpoint : {BASE_URL}")
print(f"Model    : {MODEL}")
print(f"Key      : {API_KEY[:12]}...")

In [ ]:
import httpx, json

# ── Health check ──────────────────────────────────────────────────────────────
health_url = BASE_URL.replace("/v1", "/health")
r = httpx.get(health_url, timeout=10)
print(json.dumps(r.json(), indent=2))

In [ ]:
# ── List available models ─────────────────────────────────────────────────────
r = httpx.get(BASE_URL + "/models", headers={"Authorization": f"Bearer {API_KEY}"}, timeout=10)
for m in r.json()["data"]:
    print(f"  {m['id']}  (gateway alias: {m.get('gateway_name', m['id'])})")

---
## 1. Batch (non-streaming)

Measures total round-trip latency. Use this when you want the full answer at once.

In [ ]:
import time, httpx, json

def batch(prompt, max_tokens=256, thinking=False):
    payload = {
        "model": MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "stream": False,
    }
    if thinking:
        payload["chat_template_kwargs"] = {"enable_thinking": True}

    t0 = time.perf_counter()
    r = httpx.post(
        BASE_URL + "/chat/completions",
        headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"},
        json=payload,
        timeout=120,
    )
    elapsed = time.perf_counter() - t0
    data = r.json()
    msg  = data["choices"][0]["message"]
    usage = data.get("usage", {})
    toks = usage.get("completion_tokens", 0)
    return {
        "elapsed": elapsed,
        "content": msg.get("content") or "",
        "reasoning": msg.get("reasoning") or "",
        "tokens": toks,
        "tok_per_s": toks / elapsed if elapsed > 0 else 0,
    }

# Quick sanity check
r = batch("What is the capital of France?", max_tokens=32)
print(f"Answer  : {r['content']}")
print(f"Latency : {r['elapsed']:.2f}s  |  {r['tokens']} tokens  |  {r['tok_per_s']:.1f} tok/s")

In [ ]:
# ── Batch WITH thinking (reasoning mode) ─────────────────────────────────────
MATH_PROMPT = (
    "A train travels 60 km in 40 minutes, then 90 km in 50 minutes. "
    "What is its average speed in km/h? Show your work."
)

r = batch(MATH_PROMPT, max_tokens=512, thinking=True)
print(f"=== Answer ({r['elapsed']:.2f}s, {r['tokens']} tokens, {r['tok_per_s']:.1f} tok/s) ===")
print(r["content"][:600] if r["content"] else "(no content — model still in thinking phase at max_tokens)")
print(f"\n=== Reasoning ({len(r['reasoning'])} chars) ===")
print(r["reasoning"][:600] if r["reasoning"] else "(none — thinking not triggered for this prompt)")

---
## 2. Streaming

Measures TTFT (time to first token) — this is what determines how interactive the model feels.  
Under 1 second feels real-time; 2–3 seconds is acceptable; beyond that users notice the wait.

In [ ]:
import time, json, httpx

def stream(prompt, max_tokens=256, thinking=False, print_tokens=True):
    payload = {
        "model": MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "stream": True,
    }
    if thinking:
        payload["chat_template_kwargs"] = {"enable_thinking": True}

    t0 = time.perf_counter()
    ttft = None
    content = ""
    reasoning = ""
    n_chunks = 0

    with httpx.Client(timeout=120) as client:
        with client.stream(
            "POST",
            BASE_URL + "/chat/completions",
            headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"},
            json=payload,
        ) as resp:
            for line in resp.iter_lines():
                if not line.startswith("data:"):
                    continue
                d = line[5:].strip()
                if d == "[DONE]":
                    break
                try:
                    chunk = json.loads(d)
                except json.JSONDecodeError:
                    continue
                delta = chunk["choices"][0].get("delta", {})
                tok = delta.get("content", "") or ""
                rtok = delta.get("reasoning_content", "") or ""
                if tok or rtok:
                    if ttft is None:
                        ttft = time.perf_counter() - t0
                    content += tok
                    reasoning += rtok
                    n_chunks += 1
                    if print_tokens:
                        print(tok or f"[think]{rtok}", end="", flush=True)

    total = time.perf_counter() - t0
    return {
        "ttft": ttft,
        "total": total,
        "content": content,
        "reasoning": reasoning,
        "chunks": n_chunks,
        "tok_per_s": n_chunks / total if total > 0 else 0,
    }

print("stream() helper defined.")

In [ ]:
print("=== Response ===")
r = stream("Explain the twin paradox in three sentences.", max_tokens=200)
print(f"\n\nTTFT    : {r['ttft']:.3f}s")
print(f"Total   : {r['total']:.2f}s")
print(f"Chunks  : {r['chunks']}")
print(f"Tok/s   : {r['tok_per_s']:.1f}")

In [ ]:
# reasoning_content tokens are prefixed with [think] so you can see them inline
print("=== Thinking tokens + Response ===")
r = stream(MATH_PROMPT, max_tokens=600, thinking=True)
print(f"\n\nTTFT       : {r['ttft']:.3f}s")
print(f"Total      : {r['total']:.2f}s")
print(f"Content    : {len(r['content'])} chars")
print(f"Reasoning  : {len(r['reasoning'])} chars")

---
## 3. Latency benchmark — N sequential requests

Measures TTFT distribution over multiple requests.

In [ ]:
import statistics

PROMPTS = [
    "What is 2 + 2?",
    "Name the first five elements of the periodic table.",
    "Summarize the French Revolution in one sentence.",
    "What is the derivative of x^2?",
    "Who wrote Pride and Prejudice?",
]

N_REPS = 2
results = []

for rep in range(N_REPS):
    for p in PROMPTS:
        r = stream(p, max_tokens=64, print_tokens=False)
        results.append(r)
        print(f"  TTFT={r['ttft']:.3f}s  total={r['total']:.2f}s  tok/s={r['tok_per_s']:.1f}  [{p[:40]}]")

ttfts  = [r["ttft"] for r in results]
totals = [r["total"] for r in results]
tps    = [r["tok_per_s"] for r in results]

print(f"\n{'':=<60}")
print(f"Requests : {len(results)}")
print(f"TTFT     : mean={statistics.mean(ttfts):.3f}s  p50={sorted(ttfts)[len(ttfts)//2]:.3f}s  max={max(ttfts):.3f}s")
print(f"Total    : mean={statistics.mean(totals):.2f}s")
print(f"Tok/s    : mean={statistics.mean(tps):.1f}  max={max(tps):.1f}")

---
## 4. Concurrent load test — how many users at ≤1s TTFT?

Fires N requests simultaneously and records each TTFT.  
vLLM uses continuous batching — TTFT grows as the batch fills up.

**Interpretation:**  
`✓` TTFT ≤ 1s (interactive target)  |  `~` 1–3s (acceptable)  |  `✗` > 3s (slow)

In [ ]:
import asyncio, time, json, httpx

CONCURRENT_PROMPT = "Explain the concept of entropy in thermodynamics in 2 sentences."

async def astream_ttft(client, prompt, max_tokens=128):
    payload = {"model": MODEL, "messages": [{"role": "user", "content": prompt}],
               "max_tokens": max_tokens, "stream": True}
    t0 = time.perf_counter()
    ttft = None
    n_chunks = 0
    try:
        async with client.stream(
            "POST", BASE_URL + "/chat/completions",
            headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"},
            json=payload,
        ) as resp:
            async for line in resp.aiter_lines():
                if not line.startswith("data:"): continue
                d = line[5:].strip()
                if d == "[DONE]": break
                try: chunk = json.loads(d)
                except: continue
                if chunk["choices"][0].get("delta", {}).get("content"):
                    if ttft is None: ttft = time.perf_counter() - t0
                    n_chunks += 1
    except Exception as e:
        return {"error": str(e), "ttft": None, "total": time.perf_counter() - t0}
    total = time.perf_counter() - t0
    return {"ttft": ttft, "total": total, "chunks": n_chunks,
            "tok_per_s": n_chunks / total if total > 0 else 0}


async def run_concurrent(n, prompt):
    async with httpx.AsyncClient(timeout=120) as client:
        t0 = time.perf_counter()
        results = await asyncio.gather(*[astream_ttft(client, prompt) for _ in range(n)])
        return results, time.perf_counter() - t0


for n_concurrent in [1, 2, 4, 8, 16, 32]:
    results, wall = await run_concurrent(n_concurrent, CONCURRENT_PROMPT)
    ttfts  = [r["ttft"] for r in results if r.get("ttft") is not None]
    errors = sum(1 for r in results if "error" in r)
    if ttfts:
        mean_ttft = statistics.mean(ttfts)
        p95 = sorted(ttfts)[int(len(ttfts) * 0.95)]
        sym = "\u2713" if mean_ttft <= 1.0 else ("~" if mean_ttft <= 3.0 else "\u2717")
        print(f"  n={n_concurrent:3d}  mean_TTFT={mean_ttft:.2f}s  p95={p95:.2f}s  wall={wall:.2f}s  err={errors}  {sym}")
    else:
        print(f"  n={n_concurrent:3d}  ALL ERRORS  wall={wall:.2f}s")

---
## 5. Rate limiting

The proxy enforces a per-caller sliding-window rate limit.  
- Default: `PROXY_RATE_LIMIT_REQUESTS` per window (global)  
- Per-key: `PROXY_RATE_LIMIT_REQUESTS_<NAME>` — e.g. set `PROXY_RATE_LIMIT_REQUESTS_DEMO=20` to cap the demo key at 20 req/min  

Exceeding the limit returns HTTP 429.

In [ ]:
# Fire 25 rapid 1-token requests — shows 429 if the key is rate-limited
import httpx

n_ok = 0
n_429 = 0

with httpx.Client(timeout=10) as client:
    for i in range(25):
        r = client.post(
            BASE_URL + "/chat/completions",
            headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"},
            json={"model": MODEL, "messages": [{"role": "user", "content": "hi"}],
                  "max_tokens": 1, "stream": False},
        )
        if r.status_code == 429:
            n_429 += 1
            print(f"  [{i+1:02d}] 429 — {r.json().get('detail', '')}")
        else:
            n_ok += 1

print(f"\nOK: {n_ok}  |  Rate-limited: {n_429}")

---
## 6. Capacity summary — Gemma 4 31B, 2× A100 SXM4, TP2

In [ ]:
print("""
+--------------------------+-------------------------------------------+
| Metric                   | Value                                     |
+--------------------------+-------------------------------------------+
| Decode throughput        | ~43 tok/s (single user, warm)             |
| TTFT (warm, short)       | ~70 ms (vLLM direct)                      |
| TTFT (relay, streaming)  | ~0.5-1s (network + Globus overhead)       |
| Max concurrent sessions  | 128 (--max-num-seqs)                      |
| Context window           | 128K tokens (hard requirement)            |
| KV cache at 128K         | ~85 GiB -> ~6 full-context sessions       |
+--------------------------+-------------------------------------------+
| 300 students, 10% active | ~30 concurrent -> comfortable             |
| Synchronized spike       | 100+ at once -> queue builds, TTFT grows  |
| Globus worker slots      | 4 init x 32 workers = 128 pre-warmed      |
+--------------------------+-------------------------------------------+

Advice for class use:
  - Homework / self-paced lab: students stagger naturally -> fast
  - Synchronized demos: warn students not to all submit at once
  - vLLM continuous batching: multiple students share one decode pass
""")

---
## 7. OpenAI SDK — drop-in usage

Any student can use the official `openai` library — just point it at this gateway.

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What is the capital of France?"}],
    max_tokens=32,
)
print("Non-streaming:", resp.choices[0].message.content)

print("\nStreaming: ", end="")
for chunk in client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Count from 1 to 5."}],
    max_tokens=32,
    stream=True,
):
    print(chunk.choices[0].delta.content or "", end="", flush=True)
print()

In [ ]:
# Gemma 4 thinking mode — extra_body passes through arbitrary JSON fields
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": MATH_PROMPT}],
    max_tokens=512,
    extra_body={"chat_template_kwargs": {"enable_thinking": True}},
)
msg = resp.choices[0].message
print("Content  :", (msg.content or "")[:300])
reasoning = getattr(msg, "reasoning", None) or (msg.model_extra or {}).get("reasoning", "")
print("Reasoning:", (reasoning or "")[:300])

---
## Student quick-start

Ask your instructor for the API key and endpoint URL, then:

```bash
export HPC_GATEWAY_URL="https://<ask-instructor>/v1"
export HPC_GATEWAY_KEY="sk-stream-<ask-instructor>"
```

```python
import os
from openai import OpenAI

client = OpenAI(
    base_url=os.environ["HPC_GATEWAY_URL"],
    api_key=os.environ["HPC_GATEWAY_KEY"],
)

for chunk in client.chat.completions.create(
    model="gemma4-31b",
    messages=[{"role": "user", "content": "Hello! What can you do?"}],
    stream=True,
):
    print(chunk.choices[0].delta.content or "", end="", flush=True)
```